# 3-7. Agentic RAG — LLM이 스스로 검색을 결정하는 파이프라인

🧩 심화/선택

<!-- optional -->

## 학습 목표
- 전통 RAG(고정 파이프라인)와 **agentic RAG**(LLM이 도구 호출을 주도)의 구조적 차이를 설명할 수 있다.
- LangGraph의 `create_react_agent` 또는 수동 `StateGraph`를 사용해 **tool-calling 루프**를 구현할 수 있다.
- 질문 난이도에 따라 에이전트가 검색을 **건너뛰거나 반복**하는 모습을 관찰하고, `max_iterations`로 폭주를 제어할 수 있다.
- 폐쇄망에서 agentic RAG를 사용할 때의 **감사·비용 측면 위험**을 평가할 수 있다.

## 핵심 키워드
`agentic RAG` · `tool calling` · `LangGraph ReAct` · `create_react_agent` · `multi-step retrieval` · `audit trace`

> 🧩 이 노트북은 **심화/선택** 과정이다. S3-5(Self-RAG)까지 체감했다면, 그 자가교정 루프의 결정권을 LLM에게 더 넘겨 본다.

## 1. Agentic RAG vs 전통 RAG

### 전통 RAG (단일 검색·단일 생성 파이프라인)
```
question ─▶ retriever ─▶ prompt ─▶ LLM ─▶ answer
```
- 검색은 **항상 1회**, 검색 결과는 **무조건 컨텍스트로 주입**된다.
- 장점: 예측 가능, 로그 단순, 비용 상한이 보장됨.
- 단점: "안녕하세요" 같은 일상 대화에도 검색이 돌고, 다단계 질문에는 한 번의 검색으로 부족하다.

### Agentic RAG
```
           ┌──────────────────────────────┐
           ▼                              │
  question ─▶ [LLM agent] ──▶ tool? ──▶ tool_node ──┘
                    │ no
                    ▼
                 final answer
```
- LLM이 **매 턴마다** "검색할지, 계산할지, 바로 답할지"를 tool-calling으로 결정한다.
- 장점: 쉬운 질문엔 빠르게 답하고 복잡한 질문엔 여러 번 검색해 정확도를 높인다.
- 단점: 도구 호출 폭주 위험, 감사 로그가 복잡해짐, 같은 질문도 턴마다 다른 경로를 탈 수 있음.

```mermaid
flowchart LR
    U[사용자 질문] --> A[LLM Agent]
    A -- tool_calls 없음 --> F[최종 답변]
    A -- tool_calls 있음 --> T[Tool Node]
    T --> A
```

In [ ]:
# 공통 세팅: repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '../..')

from common import get_chat_model, get_embeddings, provider_badge
print(provider_badge())

## 2. 실습용 약관 코퍼스와 retriever 준비

전자금융 표준약관 축약본을 메모리 상의 Chroma에 올린다. 이 노트북은 **단독으로 재현 가능**하도록 문서와 질문을 자기완결적으로 구성했다.

In [ ]:
from langchain_core.documents import Document

E_FINANCE_TERMS = [
    Document(page_content='제1조(목적) 본 약관은 전자금융거래에 관한 사항을 정함으로써 이용자의 권익 보호와 거래의 안전을 확보함을 목적으로 한다.', metadata={'article': '제1조'}),
    Document(page_content='제2조(정의) 전자금융거래란 금융회사 또는 전자금융업자가 전자적 장치를 통해 제공하는 금융상품 및 서비스를 자동화된 방식으로 이용하는 거래를 말한다.', metadata={'article': '제2조'}),
    Document(page_content='제5조(청약철회) ① 이용자는 계약 체결 후 14일 이내에 서면 또는 전자문서로 청약을 철회할 수 있다. ② 다만 이미 안내된 서비스가 제공되었거나 영업일 기준 3일 이내 증정이 끝난 경우 철회가 제한될 수 있다.', metadata={'article': '제5조'}),
    Document(page_content='제8조(수수료) 전자금융거래 이용 수수료는 거래 유형별로 사전에 안내되며, 수수료 인상 시 적용일 30일 전에 공지해야 한다. 기본 이체 수수료는 거래금액의 1%를 상한으로 한다.', metadata={'article': '제8조'}),
    Document(page_content='제10조(분실신고) ① 이용자는 접근매체의 분실·도난 시 즉시 고객센터 또는 가장 가까운 영업점에 신고하여야 한다. ② 신고 접수 이전에 발생한 사고에 대해서도 선의무과실이 없는 한 금융회사가 보상책임을 부담한다.', metadata={'article': '제10조'}),
    Document(page_content='제14조(이의신청) ① 이용자는 거래 내역에 이의가 있을 때 거래일로부터 1년 이내에 이의를 신청할 수 있다. ② 금융회사는 이의신청 접수일로부터 15영업일 이내에 결과를 통지한다.', metadata={'article': '제14조'}),
    Document(page_content='제17조(서비스 이용시간) 전자금융거래 서비스는 24시간 이용을 원칙으로 하되, 시스템 점검·장애·운영상 필요에 따라 일시적으로 제한될 수 있으며 사전 공지한다.', metadata={'article': '제17조'}),
]
print(f'약관 조항 {len(E_FINANCE_TERMS)}개 로드 완료')

In [ ]:
from langchain_community.vectorstores import Chroma

# 인메모리 Chroma — persist_directory를 생략하면 임시 저장소에 올라간다
vs = Chroma.from_documents(
    E_FINANCE_TERMS,
    embedding=get_embeddings(),
    collection_name='efinance_terms_agentic',
)
retriever = vs.as_retriever(search_kwargs={'k': 3})
llm = get_chat_model(temperature=0)
print('벡터스토어 및 LLM 준비 완료')

## 3. 도구 정의 — 약관 검색 · 사내 FAQ stub · 계산기

LangChain의 `@tool` 데코레이터로 에이전트가 호출할 수 있는 **3개의 도구**를 정의한다.

- `retrieve_from_terms`: 전자금융 표준약관 벡터스토어에서 관련 조항을 찾는다.
- `web_search_stub`: 폐쇄망이므로 외부 웹 검색 대신 **사내 FAQ 스텁**을 흉내 낸다. 실제 현장에서는 사내 검색엔진/다른 벡터스토어로 교체한다.
- `calculator`: 수수료 계산 등 숫자 연산을 안전히 수행한다 (LLM의 산수 환각 방지).

In [ ]:
from langchain_core.tools import tool

# 사내 FAQ 목업 — 실제론 Confluence/Sharepoint 인덱스로 교체
INTERNAL_FAQ = {
    '휴일 이체': '주말·공휴일에도 실시간 이체는 가능하나, 일부 기관은 익영업일 처리될 수 있습니다.',
    '고객센터': '금융회사 고객센터는 평일 09:00~18:00 운영되며, 분실신고는 24시간 가능합니다.',
    '비밀번호 변경': '보안상 3개월마다 비밀번호 변경을 권고합니다.',
}

@tool
def retrieve_from_terms(query: str) -> str:
    """전자금융 표준약관에서 질문과 관련된 조항을 검색해 반환한다.

    약관 해석·청약철회·수수료·분실신고 등 '규정 원문'을 묻는 질문에 사용하라.
    """
    docs = retriever.invoke(query)
    if not docs:
        return '관련 조항을 찾지 못했습니다.'
    return '\n---\n'.join(f"[{d.metadata.get('article')}] {d.page_content}" for d in docs)

@tool
def web_search_stub(keyword: str) -> str:
    """사내 FAQ(폐쇄망 스텁)에서 키워드로 간이 검색한다.

    약관에 없지만 운영 관행에 관한 질문(예: '휴일 이체 가능한가')에 사용하라.
    """
    hits = [v for k, v in INTERNAL_FAQ.items() if k in keyword or keyword in k]
    return '\n'.join(hits) if hits else '사내 FAQ에서 해당 항목을 찾지 못했습니다.'

@tool
def calculator(expression: str) -> str:
    """파이썬 산술식을 계산해 결과 문자열을 반환한다.

    예: '100000 * 0.01', '50000 + 500'. 수수료·이자 등 수치 계산에 사용하라.
    보안상 숫자·사칙연산자·괄호·점 외 문자는 거부한다.
    """
    allowed = set('0123456789+-*/(). ')
    if not set(expression) <= allowed:
        return 'ERROR: 허용되지 않은 문자가 포함되어 있습니다.'
    try:
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'ERROR: {e}'

TOOLS = [retrieve_from_terms, web_search_stub, calculator]
print('도구 3개 준비:', [t.name for t in TOOLS])

## 4. LangGraph `create_react_agent`로 에이전트 조립

LangGraph의 prebuilt ReAct 에이전트는 다음과 같은 상태 기계를 자동으로 만들어준다.

1. `agent` 노드 — LLM이 메시지 이력을 보고 tool_call을 낼지 결정한다.
2. `tools` 노드 — tool_call이 있으면 실행하고 `ToolMessage`로 결과를 추가한다.
3. 조건부 엣지 — tool_call이 더 없을 때까지 1↔2를 반복하고, 없으면 종료한다.

`recursion_limit`을 지정해 **도구 호출 폭주**를 막는다.

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = (
    '당신은 금융회사의 내부 규정을 안내하는 AI 어시스턴트입니다.\n'
    '다음 규칙을 반드시 지키세요:\n'
    '1. 약관 규정을 묻는 질문은 retrieve_from_terms 도구로 조항을 조회한 뒤 답합니다.\n'
    '2. 운영 관행·FAQ는 web_search_stub 도구를 사용합니다.\n'
    '3. 수치 계산이 필요하면 calculator 도구를 사용합니다 (절대 LLM 내부 계산 금지).\n'
    '4. 일상 대화·인사에는 도구를 호출하지 않고 곧바로 답합니다.\n'
    '5. 답변 끝에는 사용한 도구 이름을 [tools=...] 형태로 표기합니다.'
)

agent = create_react_agent(llm, tools=TOOLS, prompt=SYSTEM_PROMPT)
print('ReAct agent 조립 완료')

## 5. 실행 — 3가지 난이도의 질문

- **간단 사실**: `청약철회 기간은?` → 약관 검색 1회로 충분해야 한다.
- **계산 포함**: `10만원 거래에 수수료 1%면 실 수령액은?` → 약관 검색 + 계산기 2회 이상 호출 기대.
- **일상 대화**: `안녕하세요` → 도구 호출 없이 바로 응답해야 한다.

각 실행마다 **에이전트가 몇 번 도구를 호출했는지 집계**한다.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def run_agent(question: str, max_iterations: int = 6):
    """에이전트를 실행하고 tool-call trace를 예쁘게 출력한다."""
    print(f'\n{"=" * 70}\n❓ {question}\n{"=" * 70}')
    # recursion_limit = agent 왕복 횟수 상한 (도구 호출 폭주 방지)
    config = {'recursion_limit': max_iterations * 2}
    final = agent.invoke({'messages': [HumanMessage(content=question)]}, config=config)

    tool_calls = 0
    for msg in final['messages']:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls += 1
                args = tc.get('args', {})
                print(f'  🔧 tool_call #{tool_calls}: {tc["name"]}({args})')
        elif isinstance(msg, ToolMessage):
            preview = str(msg.content)[:80].replace('\n', ' ')
            print(f'     ↳ result: {preview}...')

    answer = final['messages'][-1].content
    print(f'\n💬 최종 답변:\n{answer}')
    print(f'\n📊 총 도구 호출: {tool_calls}회')
    return {'question': question, 'answer': answer, 'tool_calls': tool_calls}

In [ ]:
# 난이도 1: 간단 사실 — 약관 검색 1회 기대
r1 = run_agent('전자금융거래 청약철회 기간은 며칠인가요?')

In [ ]:
# 난이도 2: 계산 포함 — retrieve + calculator 호출 기대
r2 = run_agent('10만원을 이체할 때 수수료가 1%라면 실제 출금되는 총액은 얼마인가요?')

In [ ]:
# 난이도 3: 일상 대화 — 도구 호출 0회 기대
r3 = run_agent('안녕하세요')

In [ ]:
import pandas as pd

df = pd.DataFrame([r1, r2, r3])[['question', 'tool_calls']]
df.columns = ['질문', '도구 호출 횟수']
df

## 6. (부록) 수동 StateGraph로 구현해 보기

`create_react_agent`는 편리하지만 내부 동작을 이해하려면 **같은 흐름을 StateGraph로 직접 짜보는 것**이 도움이 된다. 필요할 때만 참고하라.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END, add_messages
from langgraph.prebuilt import ToolNode

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

llm_with_tools = llm.bind_tools(TOOLS)

def call_llm(state: AgentState) -> AgentState:
    # LLM이 도구 호출 여부를 결정한다
    return {'messages': [llm_with_tools.invoke(state['messages'])]}

def should_continue(state: AgentState) -> str:
    last = state['messages'][-1]
    # tool_calls가 있으면 도구 실행, 없으면 종료
    return 'tools' if getattr(last, 'tool_calls', None) else END

g = StateGraph(AgentState)
g.add_node('agent', call_llm)
g.add_node('tools', ToolNode(TOOLS))
g.set_entry_point('agent')
g.add_conditional_edges('agent', should_continue, {'tools': 'tools', END: END})
g.add_edge('tools', 'agent')  # 도구 결과를 다시 agent에게 넘긴다
manual_agent = g.compile()
print('수동 StateGraph agent 컴파일 완료')

In [ ]:
# 동작 확인 — prebuilt와 같은 질문을 넣어 본다
from langchain_core.messages import SystemMessage

out = manual_agent.invoke({
    'messages': [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content='청약철회 기간은?')],
}, config={'recursion_limit': 10})
print(out['messages'][-1].content)

## 7. 한계와 운영 시 유의사항

| 리스크 | 증상 | 완화 |
|--------|------|------|
| **도구 호출 폭주** | 같은 질문을 5~10회 재검색 | `recursion_limit`, tool-call 횟수 모니터링, grader 추가 |
| **감사 추적 어려움** | 경로가 매번 다름 | 모든 `tool_calls`를 구조화 로그로 기록, trace ID 부여 |
| **비용 예측 불가** | 한 질문에 LLM 3~5회 호출 | 일자별 호출 cap, 사용자별 토큰 쿼터 |
| **도구 오용** | calculator에 약관 본문을 넣는 등 | 각 도구의 docstring을 엄격히 작성, 검증 단계 추가 |
| **환각 누적** | 잘못된 도구 결과가 후속 질문에 영향 | tool 결과에 신뢰도 점수를 붙여 에이전트 프롬프트에 전달 |

> 🏦 금융권에서는 **"같은 입력 → 같은 출력"** 이 기본 감사 요건이다. Agentic RAG는 기본적으로 이 원칙에 반하므로,
> (1) 모든 tool_call/ToolMessage를 **불변 로그**로 저장하고,
> (2) **temperature=0 + seed 고정 + 동일 프롬프트** 조건에서 재현 가능성을 주기적으로 검사해야 한다.